In [55]:
import requests
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd
import seaborn as sns

In [56]:
API_KEY = "FGXvTQACahmE7CsypewS"


headers = {
    "auth-token": API_KEY
}
dateRange = {"start": "2022-01-01T00:00Z", "end": "2022-01-11T00:00Z"}

start_dt = pd.to_datetime(dateRange["start"], utc=True)
end_dt = pd.to_datetime(dateRange["end"], utc=True)
chunk = pd.Timedelta(days=10)
zones = ["DK-DK1", "DK-DK2"]
date_ranges = []
current_start = start_dt
while current_start < end_dt:
    current_end = min(current_start + chunk, end_dt)
    for zone in zones:
        date_ranges.append(
        (   
            zone,
            current_start.strftime("%Y-%m-%dT%H:%MZ"),
            current_end.strftime("%Y-%m-%dT%H:%MZ"),
        )
    )
    current_start = current_end


In [57]:
PDA_df = pd.DataFrame()
url = "https://api.electricitymaps.com/v4/price-day-ahead/past-range"

for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    print(response.status_code)
    print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    PDA_df = pd.concat([PDA_df, df], ignore_index=True)


PDA_df["datetime"] = pd.to_datetime(PDA_df["datetime"])
PDA_df = PDA_df.set_index("datetime")

print(PDA_df.head())

200
{"zone":"DK-DK1","data":[{"zone":"DK-DK1","datetime":"2022-01-01T00:00:00.000Z","createdAt":"2025-06-30T10:17:58.095Z","updatedAt":"2025-10-14T17:31:47.312Z","value":39.87,"unit":"EUR/MWh","source":"nordpool.com"},{"zone":"DK-DK1","datetime":"2022-01-01T01:00:00.000Z","createdAt":"2025-06-30T10:17:56.716Z","updatedAt":"2025-10-14T17:31:47.312Z","value":32.76,"unit":"EUR/MWh","source":"nordpool.com"},{"zone":"DK-DK1","datetime":"2022-01-01T02:00:00.000Z","createdAt":"2025-06-30T10:17:56.716Z","updatedAt":"2025-10-14T17:31:47.312Z","value":49.88,"unit":"EUR/MWh","source":"nordpool.com"},{"zone":"DK-DK1","datetime":"2022-01-01T03:00:00.000Z","createdAt":"2025-06-30T10:17:56.716Z","updatedAt":"2025-10-14T17:31:47.312Z","value":40.39,"unit":"EUR/MWh","source":"nordpool.com"},{"zone":"DK-DK1","datetime":"2022-01-01T04:00:00.000Z","createdAt":"2025-06-30T10:17:58.095Z","updatedAt":"2025-10-14T17:31:47.312Z","value":41.17,"unit":"EUR/MWh","source":"nordpool.com"},{"zone":"DK-DK1","datetime

In [58]:
main_df

,zone,createdAt,updatedAt,value,unit,source
datetime,,,,,,
2022-01-01 00:00:00+00:00,DK-DK1,2025-06-30T10:17:58.095Z,2025-10-14T17:31:47.312Z,39.87,EUR/MWh,nordpool.com
2022-01-01 01:00:00+00:00,DK-DK1,2025-06-30T10:17:56.716Z,2025-10-14T17:31:47.312Z,32.76,EUR/MWh,nordpool.com
2022-01-01 02:00:00+00:00,DK-DK1,2025-06-30T10:17:56.716Z,2025-10-14T17:31:47.312Z,49.88,EUR/MWh,nordpool.com
2022-01-01 03:00:00+00:00,DK-DK1,2025-06-30T10:17:56.716Z,2025-10-14T17:31:47.312Z,40.39,EUR/MWh,nordpool.com
2022-01-01 04:00:00+00:00,DK-DK1,2025-06-30T10:17:58.095Z,2025-10-14T17:31:47.312Z,41.17,EUR/MWh,nordpool.com
...,...,...,...,...,...,...
2023-12-31 19:00:00+00:00,DK-DK2,2025-06-26T12:22:06.731Z,2025-10-14T18:20:24.899Z,34.64,EUR/MWh,nordpool.com
2023-12-31 20:00:00+00:00,DK-DK2,2025-06-26T12:23:59.462Z,2025-10-14T18:20:24.899Z,29.71,EUR/MWh,nordpool.com
2023-12-31 21:00:00+00:00,DK-DK2,2025-06-26T12:22:06.731Z,2025-10-14T18:20:24.899Z,32.34,EUR/MWh,nordpool.com


In [59]:
url = "https://api.electricitymaps.com/v4/electricity-flows/past-range"
EF_df = pd.DataFrame()
for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    print(response.status_code)
    print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    EF_df = pd.concat([EF_df, df], ignore_index=True)


EF_df["datetime"] = pd.to_datetime(EF_df["datetime"])
EF_df = EF_df.set_index("datetime")

print(EF_df.head())

200
{"zone":"DK-DK1","temporalGranularity":"hourly","unit":"MW","data":[{"datetime":"2022-01-01T00:00:00.000Z","updatedAt":"2026-02-05T11:18:12.033Z","import":{"DK-DK2":381,"SE-SE3":500},"export":{"DE":530,"NL":662,"NO-NO2":927}},{"datetime":"2022-01-01T01:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","import":{"DK-DK2":318,"SE-SE3":401},"export":{"DE":671,"NL":671,"NO-NO2":997}},{"datetime":"2022-01-01T02:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","import":{"DK-DK2":249,"SE-SE3":354},"export":{"DE":422,"NL":293,"NO-NO2":1298}},{"datetime":"2022-01-01T03:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","import":{"NL":295,"DK-DK2":206,"SE-SE3":285},"export":{"DE":141,"NO-NO2":1306}},{"datetime":"2022-01-01T04:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","import":{"NL":432,"DK-DK2":389,"SE-SE3":123},"export":{"DE":109,"NO-NO2":914}},{"datetime":"2022-01-01T05:00:00.000Z","updatedAt":"2026-01-31T05:22:51.116Z","import":{"NL":401,"DK-DK2":214,"SE-SE3":119},"export":{"D

In [60]:
url = "https://api.electricitymaps.com/v4/carbon-intensity/past-range"
CI_df = pd.DataFrame()
for zone, start, end in date_ranges:
    params = {
        "zone": zone,
        "start": start,
        "end": end,
        "temporalGranularity": "hourly",
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    print(response.status_code)
    print(response.text)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["data"])
    CI_df = pd.concat([CI_df, df], ignore_index=True)


CI_df["datetime"] = pd.to_datetime(CI_df["datetime"])
CI_df = CI_df.set_index("datetime")

print(CI_df.head())

200
{"zone":"DK-DK1","data":[{"zone":"DK-DK1","carbonIntensity":149,"datetime":"2022-01-01T00:00:00.000Z","updatedAt":"2026-02-05T11:18:12.033Z","createdAt":"2025-06-30T10:17:58.095Z","emissionFactorType":"lifecycle","isEstimated":true,"estimationMethod":"SANDBOX_MODE_DATA"},{"zone":"DK-DK1","carbonIntensity":120,"datetime":"2022-01-01T01:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","createdAt":"2025-06-30T10:17:56.716Z","emissionFactorType":"lifecycle","isEstimated":true,"estimationMethod":"SANDBOX_MODE_DATA"},{"zone":"DK-DK1","carbonIntensity":175,"datetime":"2022-01-01T02:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","createdAt":"2025-06-30T10:17:56.716Z","emissionFactorType":"lifecycle","isEstimated":true,"estimationMethod":"SANDBOX_MODE_DATA"},{"zone":"DK-DK1","carbonIntensity":175,"datetime":"2022-01-01T03:00:00.000Z","updatedAt":"2026-01-27T02:52:24.481Z","createdAt":"2025-06-30T10:17:56.716Z","emissionFactorType":"lifecycle","isEstimated":true,"estimationMethod":"SAN

In [61]:
merged_df = pd.merge(PDA_df, EF_df, on="datetime", suffixes=('_price', '_flow'))
print(merged_df.head())

                             zone                 createdAt  \
datetime                                                      
2022-01-01 00:00:00+00:00  DK-DK1  2025-06-30T10:17:58.095Z   
2022-01-01 00:00:00+00:00  DK-DK1  2025-06-30T10:17:58.095Z   
2022-01-01 01:00:00+00:00  DK-DK1  2025-06-30T10:17:56.716Z   
2022-01-01 01:00:00+00:00  DK-DK1  2025-06-30T10:17:56.716Z   
2022-01-01 02:00:00+00:00  DK-DK1  2025-06-30T10:17:56.716Z   

                                    updatedAt_price  value     unit  \
datetime                                                              
2022-01-01 00:00:00+00:00  2025-10-14T17:31:47.312Z  39.87  EUR/MWh   
2022-01-01 00:00:00+00:00  2025-10-14T17:31:47.312Z  39.87  EUR/MWh   
2022-01-01 01:00:00+00:00  2025-10-14T17:31:47.312Z  32.76  EUR/MWh   
2022-01-01 01:00:00+00:00  2025-10-14T17:31:47.312Z  32.76  EUR/MWh   
2022-01-01 02:00:00+00:00  2025-10-14T17:31:47.312Z  49.88  EUR/MWh   

                                 source            updatedAt

In [62]:
merged_df = pd.merge(merged_df, CI_df, on="datetime", suffixes=('_prev', '_CI'))

In [63]:
merged_df.columns

Index(['zone_prev', 'createdAt_prev', 'updatedAt_price', 'value', 'unit',
       'source', 'updatedAt_flow', 'import', 'export', 'zone_CI',
       'carbonIntensity', 'updatedAt', 'createdAt_CI', 'emissionFactorType',
       'isEstimated', 'estimationMethod'],
      dtype='str')

In [66]:
merged_df = merged_df.drop(columns=['createdAt_prev','updatedAt_price', 'updatedAt_flow', 'createdAt_CI', 'updatedAt'])

In [67]:
merged_df.columns

Index(['zone_prev', 'value', 'unit', 'source', 'import', 'export', 'zone_CI',
       'carbonIntensity', 'emissionFactorType', 'isEstimated',
       'estimationMethod'],
      dtype='str')

In [70]:
merged_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1920 entries, 2022-01-01 00:00:00+00:00 to 2022-01-10 23:00:00+00:00
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   zone_prev           1920 non-null   str    
 1   value               1920 non-null   float64
 2   unit                1920 non-null   str    
 3   source              1920 non-null   str    
 4   import              1920 non-null   object 
 5   export              1920 non-null   object 
 6   zone_CI             1920 non-null   str    
 7   carbonIntensity     1920 non-null   int64  
 8   emissionFactorType  1920 non-null   str    
 9   isEstimated         1920 non-null   bool   
 10  estimationMethod    1920 non-null   str    
 11  day_of_week         1920 non-null   int32  
 12  month               1920 non-null   int32  
 13  hour_of_day         1920 non-null   int32  
 14  year                1920 non-null   int32  
dtypes: bool(1), float6

In [69]:
#add day of week, month, hour of day, and year as features
merged_df['day_of_week'] = merged_df.index.dayofweek
merged_df['month'] = merged_df.index.month
merged_df['hour_of_day'] = merged_df.index.hour
merged_df['year'] = merged_df.index.year


In [131]:
#Get oil price data from 

# FRED CSV endpoint for Brent Europe
FRED_ID = ["DCOILBRENTEU","PNGASEUUSDM"]
for id in FRED_ID:
    url_brent = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id={id}"
        "&cosd=2021-12-31"
        "&coed=2022-02-01"
    )
    if id == "DCOILBRENTEU":
        brent = pd.read_csv(url_brent)
        brent.set_index("observation_date", inplace=True)
        brent = brent.rename(columns={"DCOILBRENTEU": "Oil_Price"})
        brent.index = pd.to_datetime(brent.index, utc=True)
        brent = brent.resample("D").ffill()
        brent.drop("2021-12-31", inplace=True)


print(brent.head())
brent.info()

                           Oil_Price
observation_date                    
2022-01-01 00:00:00+00:00      77.24
2022-01-02 00:00:00+00:00      77.24
2022-01-03 00:00:00+00:00      78.25
2022-01-04 00:00:00+00:00      79.39
2022-01-05 00:00:00+00:00      80.60
<class 'pandas.DataFrame'>
DatetimeIndex: 32 entries, 2022-01-01 00:00:00+00:00 to 2022-02-01 00:00:00+00:00
Freq: D
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Oil_Price  32 non-null     float64
dtypes: float64(1)
memory usage: 512.0 bytes


In [132]:
import yfinance as yf

gas = yf.download("TTF=F", start="2021-12-31", end="2024-01-01")
gas.head()

# remove multi-index
gas.columns = gas.columns.get_level_values(0)

# keep only close price
gas = gas[["Close"]]

# rename
gas = gas.rename(columns={"Close": "Gas_price"})

print(gas.head())

# make sure index is datetime
gas.index = pd.to_datetime(gas.index, utc=True)

# create full daily index
gas = gas.asfreq("D")

# forward fill weekends & holidays
gas["Gas_price"] = gas["Gas_price"].ffill()

gas.drop("2021-12-31", inplace=True)


[*********************100%***********************]  1 of 1 completed

Price       Gas_price
Date                 
2021-12-31  70.343002
2022-01-03  80.433998
2022-01-04  88.740997
2022-01-05  91.522003
2022-01-06  96.501999


In [133]:
gas_oil = pd.merge(gas, brent, left_index=True, right_index=True, how="inner")
gas_oil=gas_oil.resample("h").ffill()
gas_oil.head()

,Gas_price,Oil_Price
Date,,
2022-01-01 00:00:00+00:00,70.343002,77.24
2022-01-01 01:00:00+00:00,70.343002,77.24
2022-01-01 02:00:00+00:00,70.343002,77.24
2022-01-01 03:00:00+00:00,70.343002,77.24
2022-01-01 04:00:00+00:00,70.343002,77.24


In [134]:
merged_df = pd.merge(merged_df, gas_oil, left_index=True, right_index=True, how="inner")
merged_df.head()

,zone_prev,value,unit,source,import,export,zone_CI,carbonIntensity,emissionFactorType,isEstimated,estimationMethod,day_of_week,month,hour_of_day,year,Gas_price,Oil_Price
datetime,,,,,,,,,,,,,,,,,
2022-01-01 00:00:00+00:00,DK-DK1,39.87,EUR/MWh,nordpool.com,"{'DK-DK2': 381, 'SE-SE3': 500}","{'DE': 530, 'NL': 662, 'NO-NO2': 927}",DK-DK1,149,lifecycle,True,SANDBOX_MODE_DATA,5,1,0,2022,70.343002,77.24
2022-01-01 00:00:00+00:00,DK-DK1,39.87,EUR/MWh,nordpool.com,"{'DK-DK2': 381, 'SE-SE3': 500}","{'DE': 530, 'NL': 662, 'NO-NO2': 927}",DK-DK2,213,lifecycle,True,SANDBOX_MODE_DATA,5,1,0,2022,70.343002,77.24
2022-01-01 00:00:00+00:00,DK-DK1,39.87,EUR/MWh,nordpool.com,{'DK-DK1': 429},"{'DE': 836, 'SE-SE4': 969}",DK-DK1,149,lifecycle,True,SANDBOX_MODE_DATA,5,1,0,2022,70.343002,77.24
2022-01-01 00:00:00+00:00,DK-DK1,39.87,EUR/MWh,nordpool.com,{'DK-DK1': 429},"{'DE': 836, 'SE-SE4': 969}",DK-DK2,213,lifecycle,True,SANDBOX_MODE_DATA,5,1,0,2022,70.343002,77.24
2022-01-01 01:00:00+00:00,DK-DK1,32.76,EUR/MWh,nordpool.com,"{'DK-DK2': 318, 'SE-SE3': 401}","{'DE': 671, 'NL': 671, 'NO-NO2': 997}",DK-DK1,120,lifecycle,True,SANDBOX_MODE_DATA,5,1,1,2022,70.343002,77.24


In [136]:
merged_df.to_csv("fucking_clean_ass_data_bitch.csv")

In [ ]:
api_key = '1d6c006643bb4029b352ef88d0b834c0'
dtu_lat = '55.785797' 
dtu_lon = '12.521533'

In [ ]:
url = 'https://api.weatherbit.io/v2.0/current?lat={}&lon={}&key={}'.format(dtu_lat, dtu_lon, api_key)
response = requests.get(url)
weather_data = response.json()


In [ ]:
import datetime
date_end_str = "2018-09-21"
date_end = datetime.datetime.strptime(date_end_str, '%Y-%m-%d')
n_days = 11
date_list = [date_end - datetime.timedelta(days=x) for x in range(0, n_days)]
date_list_str = [x.strftime("%Y-%m-%d") for x in date_list]
len(date_list_str)

11

In [ ]:


observations = []
for i in reversed(range(1, len(date_list_str))):
    start_date_str = date_list_str[i]
    end_date_str = date_list_str[i-1]
    url = 'https://api.weatherbit.io/v2.0/history/daily?lat={}&lon={}&start_date={}&end_date={}&key={}'
    url = url.format(dtu_lat, dtu_lon, start_date_str, end_date_str, api_key)
    contents = requests.get(url).text
    contents_json = json.loads(contents)
    observations.append(contents_json)

print(observations[0])

{'error': 'API key not valid, or not yet activated. If you recently signed up for an account or created this key, please allow up to 30 minutes for key to activate.'}
